# Career Market Intelligence — Team Notebook

**Single shared notebook.** Every team member works **inside this file, in their own section**. Run the sections in order — each one depends on the previous.

**Model target:** `salary_band` (`low` / `medium` / `high`) — India market scope.

## Sections & owners
| # | Section | Owner |
|---|---|---|
| 1 | Data Preprocessing | Member 1 |
| 2 | Exploratory Data Analysis (EDA) | Member 2 |
| 3 | Feature Engineering | Members 3 & 4 |
| 4 | Predictive Modelling | Members 3 & 4 |
| 5 | Evaluation & Business Interpretation | Member 5 |

## Data flow
```
data/final/career_market_core_analysis.csv      (input, 7,184 real postings)
    --> Section 1 --> data/processed/career_market_preprocessed.csv
    --> Sections 2-5 read the preprocessed file
```

## Working rules
- Add your work **only under your own section header**; don't edit others' cells.
- Keep cells runnable top-to-bottom (Restart & Run All should work).
- Save figures to `reports/eda/` (Section 2) where relevant.


# 1. Data Preprocessing  (Member 1)
**Owner / Responsibilty:** Member 1 (Data Preprocessing)  
**Project Goal:** Transform the raw, consolidated job market dataset (`data/final/career_market_core_analysis.csv`) into a standardized, analysis-ready table (`data/processed/career_market_preprocessed.csv`) for the downstream team.  
**Target Variable:** `salary_band` (`low` / `medium` / `high`) — India market classification scope.  

---

### Pipeline Architecture & Downstream Hand-off
```
Member 1: Preprocessing   <-- THIS NOTEBOOK (produces clean table)
Member 2: EDA & Visualization
Member 3 & 4: Feature Engineering & Predictive Modeling
Member 5: Evaluation & Business Recommendations
```

### Preprocessing Checklist & Scope
1. **Load Raw Dataset:** Read all 7,184 postings from `data/final/career_market_core_analysis.csv`.
2. **Column Selection:** Keep core analytical & modeling columns, drop 33 audit/provenance metadata fields.
3. **Target Variable Construction:** Calculate `salary_mid`, filter valid rows via `salary_passes_plausibility_screen`, generate balanced tertiles (`low`, `medium`, `high`), and set `has_salary` flag.
4. **Seniority Level Extraction:** Rule-based heuristic deriving `Entry`, `Mid`, `Senior` from job titles.
5. **Missing Value Treatment:** Impute structural missingness (`contract_time`, `location_region`, `employer` as `'unknown'`), strictly preserve target missingness without synthetic imputation, and drop empty descriptions.
6. **Text Cleaning & Lemmatization:** Strip HTML/URLs/emails, protect technical keywords (`c++`, `.net`, `node.js`), remove non-negation stopwords, and lemmatize using spaCy (`en_core_web_sm`).
7. **Standardization:** Datetime conversion, casing normalization, and deduplication verification.
8. **Export & Validation:** Output verified dataset to `data/processed/career_market_preprocessed.csv`.


In [ ]:
import re
import warnings
import pandas as pd
import numpy as np
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 100)

print("Environment setup completed.")

## Step 1: Load Raw Dataset
We load `data/final/career_market_core_analysis.csv`, which contains 7,184 rows across 49 columns.

In [ ]:
INPUT_PATH = "../data/final/career_market_core_analysis.csv"
OUTPUT_PATH = "../data/processed/career_market_preprocessed.csv"

df_raw = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"Dataset Loaded Successfully: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head(3)

## Step 2: Feature Selection (Drop Audit-Only Columns)
We retain the 16 analytical features and drop 33 columns that were used exclusively for collection provenance, deduplication audits, and scrapers.

In [ ]:
KEEP_COLS = [
    "job_id", "title", "role", "domain", "employer",
    "description_raw", "location_country", "location_region",
    "created_date", "category_label", "contract_time",
    "salary_min", "salary_max", "salary_present",
    "salary_passes_plausibility_screen", "description_length",
]

df = df_raw[KEEP_COLS].copy()
print(f"Retained {len(KEEP_COLS)} columns. Dropped {df_raw.shape[1] - len(KEEP_COLS)} audit columns.")
df.info()

## Step 3: Build the Target Variable (`salary_band`)
The classification target is **`salary_band`** (`low`, `medium`, `high`) in Indian Rupees (INR).
* **Midpoint calculation:** `salary_mid = df[['salary_min', 'salary_max']].mean(axis=1)`
* **Plausibility filter:** Salary values are restricted strictly to rows where `salary_passes_plausibility_screen == True` (1,101 rows).
* **Tertile discretization:** `pd.qcut(df.loc[has_salary, 'salary_mid'], 3, labels=['low', 'medium', 'high'])`.
* **Important rule:** Never invent or impute missing salaries for the remaining postings — they are retained for EDA and NLP tasks with `has_salary = False`.

In [ ]:
# 1. Compute midpoint (handles both ranges and fixed salaries)
df["salary_mid"] = df[["salary_min", "salary_max"]].mean(axis=1)

# 2. Invalidate non-plausible salaries
df.loc[~df["salary_passes_plausibility_screen"], "salary_mid"] = np.nan

# 3. Indicator flag for supervised modeling
df["has_salary"] = df["salary_passes_plausibility_screen"].fillna(False)

# 4. Construct 3-class target via tertiles on plausible rows
plausible_mask = df["has_salary"]
df["salary_band"] = pd.NA
df.loc[plausible_mask, "salary_band"] = pd.qcut(
    df.loc[plausible_mask, "salary_mid"],
    q=3,
    labels=["low", "medium", "high"]
)

print(f"Total postings with valid target (has_salary == True): {df['has_salary'].sum():,}")
print("\nTarget Class Distribution (salary_band):")
print(df["salary_band"].value_counts(dropna=False))

# Examine tertile cut-off points
cuts = pd.qcut(df.loc[plausible_mask, "salary_mid"], q=3).value_counts().sort_index()
print("\nTertile Boundary Intervals (INR):")
for interval, cnt in cuts.items():
    print(f"  {interval}: {cnt} rows")

## Step 4: Derive `seniority_level` Feature from Job Title
We extract seniority as a predictive feature using regex keyword patterns:
* **Senior:** `senior`, `sr`, `lead`, `principal`, `head`, `manager`, `director`
* **Entry:** `junior`, `jr`, `graduate`, `trainee`, `intern`, `entry`, `associate`
* **Mid:** default for all remaining positions

In [ ]:
SENIOR_KEYWORDS = r"\b(senior|sr\.?|lead|principal|head|manager|director)\b"
ENTRY_KEYWORDS  = r"\b(junior|jr\.?|graduate|trainee|intern|entry|associate)\b"

def derive_seniority(title: str) -> str:
    if not isinstance(title, str):
        return "Mid"
    t = title.lower()
    if re.search(SENIOR_KEYWORDS, t):
        return "Senior"
    if re.search(ENTRY_KEYWORDS, t):
        return "Entry"
    return "Mid"

df["seniority_level"] = df["title"].apply(derive_seniority)

print("Seniority Level Distribution:")
print(df["seniority_level"].value_counts())
print("\nSeniority Level vs Target (has_salary == True):")
print(pd.crosstab(df["seniority_level"], df["salary_band"], margins=True))

## Step 5: Handle Missing Values & Audit Logging
* `contract_time` (~40% missing) -> filled with `"unknown"`
* `location_region` (~34% missing) -> filled with `"unknown"`
* `employer` (~3.8% missing) -> filled with `"unknown"`
* `salary_min`, `salary_max` -> untouched (target integrity preserved)
* Empty `description_raw` -> verified and dropped if present.

In [ ]:
TRACK_COLS = ["salary_min", "salary_max", "contract_time", "location_region", "employer", "description_raw"]

print("=== MISSING VALUES BEFORE CLEANING ===")
for col in TRACK_COLS:
    cnt = df[col].isna().sum()
    pct = (cnt / len(df)) * 100
    print(f"  {col:<20}: {cnt:4d} missing ({pct:5.2f}%)")

# Impute unknown categorical labels
df["contract_time"]   = df["contract_time"].fillna("unknown")
df["location_region"] = df["location_region"].fillna("unknown")
df["employer"]        = df["employer"].fillna("unknown")

# Verify description_raw emptiness
empty_desc = df["description_raw"].isna() | (df["description_raw"].str.strip() == "")
if empty_desc.sum() > 0:
    print(f"\nDropping {empty_desc.sum()} rows with empty description_raw.")
    df = df[~empty_desc].reset_index(drop=True)
else:
    print("\nZero rows with empty description_raw.")

print("\n=== MISSING VALUES AFTER CLEANING ===")
for col in TRACK_COLS:
    cnt = df[col].isna().sum()
    pct = (cnt / len(df)) * 100
    print(f"  {col:<20}: {cnt:4d} missing ({pct:5.2f}%)")

## Step 6: Text Cleaning & Normalization (`description_raw` -> `description_clean`)
We process the job descriptions using an NLP pipeline:
1. Strip HTML tags and decode HTML entities (`&amp;`, `&lt;`, etc.).
2. Remove URLs and email addresses.
3. Protect tech tokens (e.g. `c++`, `c#`, `.net`, `node.js`, `react.js`).
4. Strip non-alphanumeric punctuation.
5. SpaCy lemmatization (`en_core_web_sm`) and stopword removal (**preserving negation words** such as `not`, `no`, `without`, `cannot`).
6. Restore tech tokens and collapse redundant whitespace.

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

NEGATIONS = {
    "not", "no", "nor", "neither", "never", "none", "nothing",
    "nowhere", "nobody", "without", "cannot",
}
STOP_WORDS_FILTERED = STOP_WORDS - NEGATIONS

TECH_TOKENS = {
    "c++":      "__CPLUSPLUS__",
    "c#":       "__CSHARP__",
    ".net":     "__DOTNET__",
    "node.js":  "__NODEJS__",
    "vue.js":   "__VUEJS__",
    "react.js": "__REACTJS__",
    "asp.net":  "__ASPNET__",
}
TECH_RESTORE = {v.lower(): k for k, v in TECH_TOKENS.items()}

def protect_tech_tokens(text: str) -> str:
    for token, placeholder in TECH_TOKENS.items():
        text = text.replace(token, placeholder)
    return text

def restore_tech_tokens(text: str) -> str:
    for placeholder, token in TECH_RESTORE.items():
        text = text.replace(placeholder, token)
    return text

def clean_description(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    # 1. HTML stripping
    text = re.sub(r"<[^>]+>", " ", text)
    for ent, repl in [("&amp;","&"),("&lt;","<"),("&gt;",">"),("&nbsp;"," "),("&quot;",'"'),("&#39;","'")]:
        text = text.replace(ent, repl)
    # 2. URLs and emails
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+\.\S+", " ", text)
    # 3. Protect tech tokens & lower
    text = protect_tech_tokens(text.lower())
    # 4. Filter characters
    text = re.sub(r"[^a-z0-9 _]", " ", text)
    # 5. Lemmatize & remove non-negation stopwords
    doc = nlp(text)
    tokens = [
        token.lemma_
        for token in doc
        if token.text not in STOP_WORDS_FILTERED
        and token.lemma_ not in STOP_WORDS_FILTERED
        and not token.is_space
        and token.text.strip()
    ]
    text = " ".join(tokens)
    # 6. Restore tech tokens
    text = restore_tech_tokens(text)
    return re.sub(r"\s+", " ", text).strip()

print("Applying NLP cleaning pipeline on description_raw...")
df["description_clean"] = df["description_raw"].apply(clean_description)
print("Text cleaning complete.")

print("\nSample Transformation (Row 0):")
print(f"RAW  : {df['description_raw'].iloc[0][:150]}...")
print(f"CLEAN: {df['description_clean'].iloc[0][:150]}...")

## Step 7: Standardization & Integrity Checks
* Convert `created_date` to pandas `datetime64[ns]`.
* Standardize string columns to `.str.strip().str.title()` (`role`, `location_region`, `location_country`, `category_label`).
* Verify uniqueness of primary key (`job_id`).

In [ ]:
# Datetime conversion
df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")

# String standardization
STR_COLS = ["role", "location_region", "location_country", "category_label"]
for col in STR_COLS:
    df[col] = df[col].astype(str).str.strip().str.title()

# Deduplication / uniqueness check
dup_count = df["job_id"].duplicated().sum()
assert dup_count == 0, f"Duplicate job_ids detected: {dup_count}"
print(f"Uniqueness check passed: {len(df):,} unique job_ids (0 duplicates).")
print(f"created_date range: {df['created_date'].min().date()} to {df['created_date'].max().date()}")

## Step 8: Final Validation, Schema Verification & Export
We arrange the final 21 columns and export to `data/processed/career_market_preprocessed.csv`.

In [ ]:
FINAL_COLS = KEEP_COLS + [
    "salary_mid", "salary_band", "has_salary",
    "seniority_level", "description_clean"
]

df_out = df[FINAL_COLS].copy()

# Validation checks
assert df_out.shape[0] == 7184, f"Row count mismatch: {df_out.shape[0]}"
assert df_out.shape[1] == 21, f"Column count mismatch: {df_out.shape[1]}"
assert df_out["job_id"].is_unique, "job_id is not unique"
assert df_out["has_salary"].sum() == 1101, f"Expected 1,101 salaried rows, got {df_out['has_salary'].sum()}"

# Export to CSV
df_out.to_csv(OUTPUT_PATH, index=False)
print(f"Preprocessed dataset saved to: {OUTPUT_PATH}")
print(f"Final Table Shape: {df_out.shape[0]:,} rows x {df_out.shape[1]} columns")

print("\n=== DEFINITION OF DONE VERIFICATION ===")
print(f" [x] career_market_preprocessed.csv exported: Yes")
print(f" [x] 7,184 rows preserved & job_id unique:    Yes ({len(df_out):,} rows)")
print(f" [x] salary_band built on plausible rows:      Yes ({df_out['has_salary'].sum():,} rows)")
print(f" [x] has_salary flag present:                   Yes (True: {df_out['has_salary'].sum()}, False: {(~df_out['has_salary']).sum()})")
print(f" [x] seniority_level derived:                   Yes (Entry: {(df_out['seniority_level']=='Entry').sum()}, Mid: {(df_out['seniority_level']=='Mid').sum()}, Senior: {(df_out['seniority_level']=='Senior').sum()})")
print(f" [x] description_clean created:                 Yes")
print(f" [x] No synthetic salary values invented:       Yes (Missing salary remains NaN)")

## Hand-off Summary for Team Members
* **To Member 2 (EDA):** The clean table is available at `data/processed/career_market_preprocessed.csv`. All 7,184 rows are available for market distribution, role trends, and location analysis.
* **To Members 3 & 4 (Feature Engineering & Modeling):**
  * Train the classifier only on rows where `has_salary == True` (1,101 rows).
  * Target variable: `salary_band` (`low`, `medium`, `high`).
  * Features available: `role`, `domain`, `employer`, `location_region`, `contract_time`, `seniority_level`, and `description_clean`.


---
# 2. Exploratory Data Analysis (EDA)  (Member 2)

Reads `data/processed/career_market_preprocessed.csv`. Follow `docs/eda_plan.md`.
Analyse the salary-band target, salary-vs-role/seniority/location, market overview, text/skills (note 500-char truncation), and data quality. Save charts to `reports/eda/`.

*Add your EDA cells below.*

## 2.1 EDA Setup and Dataset Overview

This section explores the cleaned career-market dataset to understand:

- Salary-band distribution
- Salary characteristics
- Role, seniority, category and location distributions
- Contract patterns
- Job-description length and common terms
- Skill prevalence
- Data quality, missing values and salary outliers
- Relationships between salary and selected categorical variables

Salary-related analysis is restricted to records where `has_salary == True` (1,101 records). General market distributions use all 7,184 records.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter

EDA_DIR = "../reports/eda"
os.makedirs(EDA_DIR, exist_ok=True)

INPUT_PATH = "../data/processed/career_market_preprocessed.csv"

eda_df = pd.read_csv(INPUT_PATH, low_memory=False)

print("Dataset shape:", eda_df.shape)
print("Salary records:", eda_df["has_salary"].sum())
print("Non-salary records:", (~eda_df["has_salary"]).sum())

## 2.2 Dataset Overview

In [ ]:
overview = pd.DataFrame({
    "rows": [len(eda_df)],
    "columns": [eda_df.shape[1]],
    "unique_job_ids": [eda_df["job_id"].nunique()],
    "salary_records": [eda_df["has_salary"].sum()],
    "salary_percentage": [eda_df["has_salary"].mean() * 100]
})

display(overview.round(2))

## 2.3 Salary Band Distribution

The target variable `salary_band` is available only for records with salary information. The distribution is examined using the 1,101 salaried records.

In [ ]:
salary_df = eda_df[eda_df["has_salary"] == True].copy()

salary_band_counts = (
    salary_df["salary_band"]
    .value_counts()
    .reindex(["low", "medium", "high"])
)

print(salary_band_counts)

plt.figure(figsize=(7, 5))
salary_band_counts.plot(kind="bar")
plt.title("Salary Band Distribution")
plt.xlabel("Salary Band")
plt.ylabel("Number of Jobs")
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/01_salary_band_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2.4 Salary Distribution and Summary Statistics

In [ ]:
salary_stats = salary_df["salary_mid"].describe()

print(salary_stats)

print("\nMedian salary:", salary_df["salary_mid"].median())
print("Q1:", salary_df["salary_mid"].quantile(0.25))
print("Q3:", salary_df["salary_mid"].quantile(0.75))

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(salary_df["salary_mid"].dropna(), bins=30)

plt.title("Distribution of Salary Midpoints")
plt.xlabel("Salary Midpoint (INR)")
plt.ylabel("Number of Jobs")
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/02_salary_mid_histogram.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

plt.boxplot(salary_df["salary_mid"].dropna(), vert=False)

plt.title("Salary Midpoint Boxplot")
plt.xlabel("Salary Midpoint (INR)")
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/03_salary_mid_boxplot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2.5 Overall Market Distribution

In [ ]:
role_counts = eda_df["role"].value_counts().head(15)

plt.figure(figsize=(9, 6))
role_counts.sort_values().plot(kind="barh")

plt.title("Top 15 Job Roles")
plt.xlabel("Number of Jobs")
plt.ylabel("Role")
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/04_top_roles.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
seniority_counts = eda_df["seniority_level"].value_counts()

plt.figure(figsize=(7, 5))
seniority_counts.plot(kind="bar")

plt.title("Job Distribution by Seniority Level")
plt.xlabel("Seniority Level")
plt.ylabel("Number of Jobs")
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/05_seniority_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
category_counts = eda_df["category_label"].value_counts().head(15)

plt.figure(figsize=(9, 6))
category_counts.sort_values().plot(kind="barh")

plt.title("Top Job Categories")
plt.xlabel("Number of Jobs")
plt.ylabel("Category")
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/06_category_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
contract_counts = eda_df["contract_time"].value_counts()

plt.figure(figsize=(8, 5))
contract_counts.plot(kind="bar")

plt.title("Contract Type Distribution")
plt.xlabel("Contract Type")
plt.ylabel("Number of Jobs")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/07_contract_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2.6 Location Distribution

In [ ]:
region_counts = eda_df["location_region"].value_counts().head(15)

plt.figure(figsize=(9, 6))
region_counts.sort_values().plot(kind="barh")

plt.title("Top 15 Job Locations by Region")
plt.xlabel("Number of Jobs")
plt.ylabel("Region")
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/08_location_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2.7 Salary vs Job Role

Only salaried records are used. The analysis focuses on the 10 roles with the highest number of salary observations to avoid unstable estimates from very small groups.

In [ ]:
top_salary_roles = (
    salary_df["role"]
    .value_counts()
    .head(10)
    .index
)

role_salary_df = salary_df[
    salary_df["role"].isin(top_salary_roles)
].copy()

role_order = (
    role_salary_df.groupby("role")["salary_mid"]
    .median()
    .sort_values()
    .index
)

plt.figure(figsize=(10, 7))

role_salary_df.boxplot(
    column="salary_mid",
    by="role",
    grid=False,
    rot=45
)

plt.title("Salary Distribution by Major Job Roles")
plt.suptitle("")
plt.xlabel("Role")
plt.ylabel("Salary Midpoint (INR)")
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/09_salary_by_role.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2.8 Salary vs Seniority

In [ ]:
seniority_salary = (
    salary_df.groupby("seniority_level")["salary_mid"]
    .agg(["count", "median", "mean"])
    .sort_values("median")
)

display(seniority_salary.round(2))

In [ ]:
seniority_order = (
    salary_df.groupby("seniority_level")["salary_mid"]
    .median()
    .sort_values()
    .index
)

data = [
    salary_df.loc[
        salary_df["seniority_level"] == level,
        "salary_mid"
    ].dropna()
    for level in seniority_order
]

plt.figure(figsize=(9, 6))

plt.boxplot(
    data,
    tick_labels=list(seniority_order)
)

plt.title("Salary Distribution by Seniority Level")
plt.xlabel("Seniority Level")
plt.ylabel("Salary Midpoint (INR)")
plt.xticks(rotation=30)
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/10_salary_by_seniority.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
## 2.9 Salary vs Location

In [ ]:
top_salary_regions = (
    salary_df["location_region"]
    .value_counts()
    .head(10)
    .index
)

region_salary_df = salary_df[
    salary_df["location_region"].isin(top_salary_regions)
]

plt.figure(figsize=(10, 7))

region_salary_df.boxplot(
    column="salary_mid",
    by="location_region",
    grid=False,
    rot=45
)

plt.title("Salary Distribution by Major Job Locations")
plt.suptitle("")
plt.xlabel("Location Region")
plt.ylabel("Salary Midpoint (INR)")
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/11_salary_by_location.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
## 2.10 Salary vs Job Category

In [ ]:
top_salary_categories = (
    salary_df["category_label"]
    .value_counts()
    .head(10)
    .index
)

category_salary_df = salary_df[
    salary_df["category_label"].isin(top_salary_categories)
]

plt.figure(figsize=(10, 7))

category_salary_df.boxplot(
    column="salary_mid",
    by="category_label",
    grid=False,
    rot=45
)

plt.title("Salary Distribution by Job Category")
plt.suptitle("")
plt.xlabel("Category")
plt.ylabel("Salary Midpoint (INR)")
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/12_salary_by_category.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
## 2.11 Salary-Band Composition

In [ ]:
role_band = pd.crosstab(
    salary_df["role"],
    salary_df["salary_band"],
    normalize="index"
)

top_roles = salary_df["role"].value_counts().head(10).index

role_band.loc[top_roles].plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6)
)

plt.title("Salary-Band Composition for Major Roles")
plt.xlabel("Role")
plt.ylabel("Proportion")
plt.xticks(rotation=45)
plt.legend(title="Salary Band")
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/13_role_salary_band_composition.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
## 2.12 Job Description Length

In [ ]:
print(eda_df["description_length"].describe())

plt.figure(figsize=(8, 5))

plt.hist(
    eda_df["description_length"],
    bins=30
)

plt.title("Distribution of Job Description Length")
plt.xlabel("Description Length (characters)")
plt.ylabel("Number of Jobs")
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/14_description_length.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2.13 Common Terms in Job Descriptions

The cleaned descriptions are used for exploratory term-frequency analysis. This is descriptive text analysis only and does not perform feature engineering.

In [ ]:
all_text = " ".join(
    eda_df["description_clean"]
    .dropna()
    .astype(str)
)

words = re.findall(r"\b[a-zA-Z][a-zA-Z0-9+#.-]*\b", all_text.lower())

word_counts = Counter(words)

top_words = pd.DataFrame(
    word_counts.most_common(20),
    columns=["term", "count"]
)

display(top_words)

In [ ]:
plt.figure(figsize=(9, 7))

top_words.sort_values("count").plot(
    x="term",
    y="count",
    kind="barh",
    legend=False,
    figsize=(9, 7)
)

plt.title("Top 20 Terms in Job Descriptions")
plt.xlabel("Frequency")
plt.ylabel("Term")
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/15_top_terms.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2.14 Skill Prevalence

Skill prevalence is estimated using keyword matching in `description_clean`. Since the preprocessing pipeline truncates job descriptions to 500 characters, these counts represent a lower-bound estimate of skill mentions in the original job postings.

In [ ]:
skills = [
    "python",
    "sql",
    "excel",
    "power bi",
    "aws",
    "java",
    "communication"
]

skill_results = []

text = eda_df["description_clean"].fillna("").astype(str).str.lower()

for skill in skills:
    count = text.str.contains(
        rf"\b{re.escape(skill)}\b",
        regex=True
    ).sum()

    skill_results.append({
        "skill": skill,
        "job_count": count,
        "percentage": count / len(eda_df) * 100
    })

skill_df = pd.DataFrame(skill_results)

display(skill_df.round(2))

In [ ]:
plt.figure(figsize=(9, 6))

plt.bar(
    skill_df["skill"],
    skill_df["percentage"]
)

plt.title("Prevalence of Selected Skills")
plt.xlabel("Skill")
plt.ylabel("Percentage of Job Postings")
plt.xticks(rotation=30)
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/16_skill_prevalence.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2.15 Data Quality — Missing Values

In [ ]:
missing = (
    eda_df.isna()
    .sum()
    .sort_values(ascending=False)
)

missing = missing[missing > 0]

display(
    pd.DataFrame({
        "missing_count": missing,
        "missing_percentage": missing / len(eda_df) * 100
    }).round(2)
)

In [ ]:
plt.figure(figsize=(9, 6))

missing.sort_values().plot(kind="barh")

plt.title("Missing Values by Column")
plt.xlabel("Number of Missing Values")
plt.ylabel("Column")
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/17_missing_values.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2.16 Salary Outlier Check

The IQR rule is used only to identify potentially extreme salary midpoint observations. These observations are not removed or modified during EDA.

In [ ]:
q1 = salary_df["salary_mid"].quantile(0.25)
q3 = salary_df["salary_mid"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = salary_df[
    (salary_df["salary_mid"] < lower_bound) |
    (salary_df["salary_mid"] > upper_bound)
]

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Potential salary outliers:", len(outliers))

## 2.17 Duplicate Job ID Check

In [ ]:
duplicate_count = eda_df["job_id"].duplicated().sum()

print("Duplicate job IDs:", duplicate_count)

## 2.18 Numeric Correlation

In [ ]:
numeric_cols = [
    "salary_mid",
    "description_length"
]

correlation = eda_df[numeric_cols].corr()

display(correlation.round(3))

In [ ]:
plt.figure(figsize=(6, 5))

plt.imshow(correlation, aspect="auto")

plt.xticks(
    range(len(correlation.columns)),
    correlation.columns,
    rotation=30
)

plt.yticks(
    range(len(correlation.index)),
    correlation.index
)

plt.colorbar(label="Correlation")

plt.title("Correlation Between Selected Numeric Variables")
plt.tight_layout()

plt.savefig(
    f"{EDA_DIR}/18_numeric_correlation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2.19 Key EDA Findings

- The dataset contains 7,184 job postings, of which 1,101 contain usable salary information and 6,083 do not.
- The available salary records are distributed across the low, medium and high salary bands; the exact counts are reported in the salary-band analysis above.
- Salary distributions vary across job roles, seniority levels, locations and job categories, with differences visible in the corresponding boxplots.
- The overall market is concentrated in a subset of frequently occurring roles, categories and locations.
- Job descriptions vary in length, and the most frequently occurring terms provide an overview of the language used in the postings.
- Selected skill keywords such as Python, SQL, Excel, Power BI, AWS, Java and communication occur at different frequencies across the job postings.
- The dataset contains no duplicate `job_id` values.
- Salary outlier analysis identifies potentially extreme salary midpoint observations using the IQR rule; these observations are retained for downstream analysis.
- Skill-frequency results should be interpreted as lower-bound estimates because the preprocessing pipeline truncates job descriptions to 500 characters.

---
# 3. Feature Engineering  (Members 3 & 4)

Build model inputs from the preprocessed data: skill flags from `description_clean`, encode categoricals (`role`, `seniority_level`, `location_region`, `category_label`), scale numerics, and prepare the modelling matrix (filter to `has_salary == True` for the salary-band model).

*Add your feature-engineering cells below.*

---
# 4. Predictive Modelling  (Members 3 & 4)

Train the salary-band classifier (`low` / `medium` / `high`) on the `has_salary == True` rows. Use a train/test split with stratification; start with interpretable models (Logistic Regression, Random Forest).

*Add your modelling cells below.*

---
# 5. Evaluation & Business Interpretation  (Member 5)

Report accuracy / F1 / confusion matrix, feature importance ("what drives pay"), and the business takeaways. State the India-scope and salary-disclosure-bias limitations.

*Add your evaluation cells below.*